In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "README.md").is_file() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").is_file():
    raise FileNotFoundError("Run this notebook from the project directory or one of its subdirectories.")


# EXP26 — iTransformer + Pure RMSE Loss Engine & Causal Test Inference

EXP_LOSS 실험에서 검증된 최적 손실 함수 **`PureRMSE`**를 기본 엔진으로 적용하여 학습의 안정성과 피크 일반화 성능을 극대화한 파이프라인입니다.

### 핵심 구현 내역
1. **PureRMSE Loss**: 불필요한 인위적 가중치를 제거하고 순수 RMSE 손실 함수 적용 (그래디언트 안정화)
2. **Comp-RMSE Target Early Stopping**: 검증셋의 $h_s \ge 1.5\text{m}$ 피크 지표(`comp_rmse`)를 기준으로 조기 종료 및 Best Checkpoint 저장
3. **Causal Test Feature Preprocessing**: 테스트 셋에서 `bfill`을 차단하고 `ffill`과 Train 통계치 기반 fallback으로 누수 없는 안정 추론
4. **자동 Optuna HPO & 최종 모델 재학습 및 제출 파일(`submission_exp26.csv`) 생성**

In [ ]:
# ============================================================
# 0. SETUP & PACKAGES
# ============================================================
!pip -q install optuna

from pathlib import Path
import gc
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================
DATA_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_physics_v6.csv")
TEST_CONTEXT_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_context.parquet")
TEST_INDEX_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_index.csv")
SUBMISSION_PATH = Path(PROJECT_ROOT / "submission" / "submission_exp26.csv")

EXP_DIR = PROJECT_ROOT / 'artifacts' / 'experiments' / 'exp26'
EXP_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = 'time'
STATION_COL = 'station'
STEP_MINUTES = 10
STEPS_PER_HOUR = 6

INPUT_LEN = 289
LEAD_HOURS = [3, 6, 9, 12, 18, 24]
LEAD_STEPS = [h * STEPS_PER_HOUR for h in LEAD_HOURS]
MAX_LEAD = max(LEAD_STEPS)
N_TARGETS = len(LEAD_STEPS)

TRAIN_RATIO = 0.80
N_TRIALS = 20
OPTUNA_EPOCHS = 15
OPTUNA_PATIENCE = 4
FINAL_EPOCHS = 35
FINAL_PATIENCE = 6
NUM_WORKERS = 0

print('INPUT_LEN:', INPUT_LEN)
print('LEADS:', LEAD_HOURS)


In [ ]:
# ============================================================
# 2. LOAD DATA & FEATURES (STORM_PHYSICS)
# ============================================================
df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)

if 'hs_original_observed' not in df.columns:
    df['hs_original_observed'] = df['hs'].notna().astype(np.int8)

# 검증된 최적 피처 세트 (20개 피처)
BEST_FEATURES = [
    'hs', 'tp', 'hmax', 'wspd', 'gust', 'u_wind', 'v_wind',
    'wspd_mean_6h', 'wspd_mean_12h', 'gust_max_6h', 'gust_max_12h',
    'gust_minus_wspd', 'caph_change_3h', 'caph_change_6h', 'caph_change_12h',
    'wave_steepness', 'wave_energy', 'effective_wind_forcing', 'u_wave', 'v_wave'
]
BEST_FEATURES = [c for c in BEST_FEATURES if c in df.columns]
print(f'BEST_FEATURES ({len(BEST_FEATURES)} cols):', BEST_FEATURES)

STATION_TO_ID = {st: i for i, st in enumerate(sorted(df[STATION_COL].unique()))}
WINDOW_META_KEYS = ('station_id', 'start_idx', 'end_idx', 'origin_hs', 'origin_time')

def build_samples(frame, features, input_len=INPUT_LEN, lead_steps=LEAD_STEPS):
    station_id, start_idx, end_idx, origin_hs, origin_time = [], [], [], [], []
    x_by_station, hs_by_station = {}, {}
    for station, g in frame.groupby(STATION_COL, sort=False):
        g = g.sort_values(TIME_COL).reset_index(drop=True)
        sid = STATION_TO_ID[station]
        Xv = g.loc[:, features].to_numpy(dtype=np.float32, copy=True)
        hsv = g['hs'].to_numpy(dtype=np.float32, copy=True)
        obs = g['hs_original_observed'].to_numpy(dtype=np.int8, copy=True)
        times = g[TIME_COL].to_numpy(copy=True)
        x_by_station[sid] = Xv
        hs_by_station[sid] = hsv
        dt = pd.Series(g[TIME_COL]).diff().dt.total_seconds().div(60).to_numpy()
        for e in range(input_len - 1, len(g) - max(lead_steps)):
            s = e - input_len + 1
            if not np.all(dt[s + 1:e + 1] == STEP_MINUTES): continue
            if not np.isfinite(Xv[s:e + 1]).all(): continue
            target_idx = np.asarray([e + step for step in lead_steps], dtype=np.int64)
            if not np.isfinite(hsv[target_idx]).all() or not np.all(obs[target_idx] == 1): continue
            if not np.isfinite(hsv[e]): continue
            station_id.append(sid); start_idx.append(s); end_idx.append(e)
            origin_hs.append(hsv[e]); origin_time.append(times[e])
    return {
        'station_id': np.asarray(station_id, dtype=np.int64),
        'start_idx': np.asarray(start_idx, dtype=np.int64),
        'end_idx': np.asarray(end_idx, dtype=np.int64),
        'origin_hs': np.asarray(origin_hs, dtype=np.float32),
        'origin_time': np.asarray(origin_time, dtype='datetime64[ns]'),
        'X_by_station': x_by_station, 'hs_by_station': hs_by_station,
        'source_frame': frame, 'features': list(features), 'n_features': len(features),
    }

def chronological_split(samples, train_ratio=TRAIN_RATIO):
    times = pd.Series(pd.to_datetime(samples['origin_time']))
    source_tz = samples['source_frame'][TIME_COL].dt.tz
    current_tz = times.dt.tz
    if source_tz is not None:
        times = times.dt.tz_localize(source_tz) if current_tz is None else times.dt.tz_convert(source_tz)
    elif current_tz is not None:
        times = times.dt.tz_localize(None)
    cutoff = times.quantile(train_ratio)
    train_mask = (times <= cutoff).to_numpy()
    valid_mask = (times > cutoff).to_numpy()
    def subset(mask):
        part = {key: samples[key][mask] for key in WINDOW_META_KEYS}
        part.update({
            'X_by_station': samples['X_by_station'], 'hs_by_station': samples['hs_by_station'],
            'source_frame': samples['source_frame'], 'features': samples['features'],
            'n_features': samples['n_features'], 'split_cutoff': cutoff,
        })
        return part
    return subset(train_mask), subset(valid_mask), cutoff

def scale_samples(train, valid):
    scaler = StandardScaler()
    train_rows = train['source_frame'][TIME_COL] <= train['split_cutoff']
    scaler.fit(train['source_frame'].loc[train_rows, train['features']])
    for Xv in train['X_by_station'].values():
        scaler.transform(Xv, copy=False)
    return dict(train), dict(valid), scaler

class WaveDataset(Dataset):
    def __init__(self, samples):
        self.X_by_station = samples['X_by_station']
        self.hs_by_station = samples['hs_by_station']
        self.station_id = samples['station_id']
        self.start_idx = samples['start_idx']
        self.end_idx = samples['end_idx']
        self.origin_hs = samples['origin_hs']
        self.lead_steps = np.asarray(LEAD_STEPS, dtype=np.int64)
    def __len__(self): return len(self.end_idx)
    def __getitem__(self, idx):
        sid = int(self.station_id[idx])
        start, end = int(self.start_idx[idx]), int(self.end_idx[idx])
        X = torch.from_numpy(self.X_by_station[sid][start:end + 1])
        y = torch.from_numpy(self.hs_by_station[sid][end + self.lead_steps])
        return X, y, torch.tensor(self.origin_hs[idx], dtype=torch.float32), torch.tensor(sid)

def make_loader(samples, batch_size=128, shuffle=False):
    ds = WaveDataset(samples)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=False, drop_last=False)

raw_samples = build_samples(df, BEST_FEATURES)
train_samples, valid_samples, split_cutoff = chronological_split(raw_samples)
train_samples, valid_samples, best_scaler = scale_samples(train_samples, valid_samples)
print('Train samples:', len(train_samples['end_idx']), '| Valid samples:', len(valid_samples['end_idx']))


In [ ]:
# ============================================================
# 3. MODEL ARCHITECTURE & PURE RMSE LOSS ENGINE
# ============================================================
class ITransformer(nn.Module):
    def __init__(self, seq_len, n_features, pred_len, d_model=64, n_heads=2, e_layers=4, dropout=0.25, d_ff=None):
        super().__init__()
        self.seq_len = seq_len; self.n_features = n_features; self.pred_len = pred_len
        if d_ff is None: d_ff = d_model * 4
        self.value_embedding = nn.Linear(seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation='gelu', norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=e_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(n_features * d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, pred_len)
        )
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.value_embedding(x)
        x = self.encoder(x)
        x = self.norm(x)
        return self.head(x)

# [EXP26 핵심]: 검증된 PureRMSE Loss 적용
class PureRMSELoss(nn.Module):
    def forward(self, pred, target, origin_hs):
        return torch.sqrt(torch.mean((pred - target) ** 2) + 1e-6)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def lead_rmse(y_true, y_pred):
    return {f'rmse_{h}h': rmse(y_true[:, i], y_pred[:, i]) for i, h in enumerate(LEAD_HOURS)}

def competition_like_rmse(y_true, y_pred, origin_hs):
    mask = origin_hs >= 1.5
    if mask.sum() == 0: return np.nan, 0
    return rmse(y_true[mask], y_pred[mask]), int(mask.sum())

def evaluate_model(model, loader):
    model.eval()
    preds, ys, origins = [], [], []
    with torch.no_grad():
        for X, y, origin_hs, _ in loader:
            X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            preds.append(model(X).cpu().numpy())
            ys.append(y.cpu().numpy())
            origins.append(origin_hs.numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    origin_hs = np.concatenate(origins)
    comp, comp_n = competition_like_rmse(y_true, y_pred, origin_hs)
    result = {'overall_rmse': rmse(y_true, y_pred), 'comp_rmse': comp, 'comp_valid_n': comp_n, **lead_rmse(y_true, y_pred)}
    return result, y_true, y_pred

def train_one_model(train_samples, valid_samples, params, max_epochs, patience, seed=42, verbose=True):
    seed_everything(seed)
    train_loader = make_loader(train_samples, batch_size=params.get('batch_size', 128), shuffle=True)
    valid_loader = make_loader(valid_samples, batch_size=params.get('batch_size', 128), shuffle=False)

    model = ITransformer(
        seq_len=INPUT_LEN, n_features=train_samples['n_features'], pred_len=N_TARGETS,
        d_model=params['d_model'], n_heads=params['n_heads'],
        e_layers=params['e_layers'], dropout=params['dropout'], d_ff=params.get('d_ff', params['d_model'] * 4)
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    criterion = PureRMSELoss()

    best_state, best_score, wait = None, np.inf, 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for X, y, origin_hs, _ in train_loader:
            X, y, origin_hs = X.to(DEVICE), y.to(DEVICE), origin_hs.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X), y, origin_hs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        metrics, _, _ = evaluate_model(model, valid_loader)
        # Early stopping: 대회 지표인 comp_rmse 기준으로 최적점 포획
        score = metrics['comp_rmse']
        history.append({'epoch': epoch, 'train_loss': float(np.mean(train_losses)), **metrics})
        if verbose:
            print(f'Epoch {epoch:02d} | train={np.mean(train_losses):.5f} | overall={metrics["overall_rmse"]:.5f} | comp={metrics["comp_rmse"]:.5f}')

        if score < best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience: break

    if best_state is not None:
        model.load_state_dict(best_state)
    final_metrics, y_true, y_pred = evaluate_model(model, valid_loader)
    return model, final_metrics, pd.DataFrame(history), y_true, y_pred


In [ ]:
# ============================================================
# 4. OPTUNA HPO (TUNED FOR PURE RMSE)
# ============================================================
def objective(trial):
    d_model = trial.suggest_categorical('d_model', [64, 128])
    params = {
        'd_model': d_model,
        'n_heads': trial.suggest_categorical('n_heads', [2, 4]),
        'e_layers': trial.suggest_int('e_layers', 3, 4),
        'dropout': trial.suggest_float('dropout', 0.15, 0.35),
        'lr': trial.suggest_float('lr', 1.0e-5, 4.0e-5, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [64, 128]),
    }
    try:
        m, metrics, _, _, _ = train_one_model(train_samples, valid_samples, params=params, max_epochs=OPTUNA_EPOCHS, patience=OPTUNA_PATIENCE, verbose=False)
        score = metrics['comp_rmse']
        del m; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return score
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect(); raise optuna.TrialPruned('CUDA OOM')
        raise

print('Starting Optuna HPO with PureRMSE Loss...')
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)
print('BEST comp_rmse:', study.best_value)
print('BEST PARAMS:', study.best_params)


In [ ]:
# ============================================================
# 5. FINAL RETRAINING WITH BEST PARAMS
# ============================================================
BEST_PARAMS = dict(study.best_params)
print('\n--- Final Model Retraining ---')
final_model, final_metrics, final_history, y_true, y_pred = train_one_model(
    train_samples, valid_samples, params=BEST_PARAMS, max_epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE, verbose=True
)

print('\n===== FINAL EXP26 METRICS =====')
display(pd.Series(final_metrics).to_frame('value'))


In [ ]:
# ============================================================
# 6. CAUSAL TEST INFERENCE & FINAL SUBMISSION
# ============================================================
BASE_FALLBACK_COLS = ['hs', 'tp', 'wspd', 'gust', 'wdir', 'wvdir', 'airt', 'relh', 'caph']
station_medians = df.groupby('station')[BASE_FALLBACK_COLS].median()
global_medians = df[BASE_FALLBACK_COLS].median()

ratio_df = df[df['hs'].notna() & df['hmax'].notna() & (df['hs'] > 0)].copy()
ratio_df['ratio'] = ratio_df['hmax'] / ratio_df['hs']
station_hmax_ratio = ratio_df[ratio_df['ratio'].between(1.0, 3.0)].groupby('station')['ratio'].median()
global_hmax_ratio = float(ratio_df['ratio'].median())

def get_station_median(station, col):
    val = station_medians.loc[station, col] if (station in station_medians.index and col in station_medians.columns) else np.nan
    return float(val) if np.isfinite(val) else float(global_medians[col])

def get_hmax_ratio(station):
    val = station_hmax_ratio.loc[station] if station in station_hmax_ratio.index else np.nan
    return float(val) if np.isfinite(val) else global_hmax_ratio

def add_test_features_exp26(context):
    feature_frames = []
    for case_id, group in context.groupby('case_id', sort=False):
        g = group.sort_values('step_minute').copy()
        st = g['station'].iloc[0]

        for col in ['hs', 'tp', 'hmax']: g.loc[g[col] <= 0, col] = np.nan
        for col in ['wspd', 'gust']: g.loc[g[col] < 0, col] = np.nan
        g.loc[~g['relh'].between(0, 100), 'relh'] = np.nan
        g.loc[~g['caph'].between(950, 1050), 'caph'] = np.nan

        num_cols = ['hs', 'tp', 'wspd', 'gust', 'airt', 'relh', 'caph']
        g[num_cols] = g[num_cols].ffill()
        for col in num_cols:
            if g[col].isna().any():
                g[col] = g[col].fillna(get_station_median(st, col))

        for col in ['wdir', 'wvdir']:
            g[col] = g[col].ffill()
            if g[col].isna().any():
                g[col] = g[col].fillna(get_station_median(st, col))
            g[col] %= 360.0

        hmax_ratio = get_hmax_ratio(st)
        g['hmax'] = g['hmax'].fillna(g['hs'] * hmax_ratio)
        g['hs'] = g['hs'].clip(lower=0.01)
        g['tp'] = g['tp'].clip(lower=0.01)
        g['wspd'] = g['wspd'].clip(lower=0.0)
        g['gust'] = np.maximum(g['gust'].clip(lower=0.0), g['wspd'])
        g['hmax'] = np.maximum(g['hmax'], g['hs'])

        wdir_rad = np.deg2rad(g['wdir'])
        wvdir_rad = np.deg2rad(g['wvdir'])
        g['u_wind'] = g['wspd'] * np.sin(wdir_rad)
        g['v_wind'] = g['wspd'] * np.cos(wdir_rad)
        g['u_wave'] = g['hs'] * np.sin(wvdir_rad)
        g['v_wave'] = g['hs'] * np.cos(wvdir_rad)

        diff = ((g['wdir'] - g['wvdir'] + 180) % 360) - 180
        g['wind_wave_diff'] = np.abs(diff)
        g['wind_wave_alignment'] = np.cos(np.deg2rad(diff))

        g['hs_diff_1h'] = g['hs'] - g['hs'].shift(6)
        g['hs_diff_3h'] = g['hs'] - g['hs'].shift(18)
        g['hs_mean_6h'] = g['hs'].rolling(36, min_periods=1).mean()
        g['hs_mean_12h'] = g['hs'].rolling(72, min_periods=1).mean()
        g['hs_max_6h'] = g['hs'].rolling(36, min_periods=1).max()
        g['hs_max_12h'] = g['hs'].rolling(72, min_periods=1).max()

        g['wspd_mean_6h'] = g['wspd'].rolling(36, min_periods=1).mean()
        g['wspd_mean_12h'] = g['wspd'].rolling(72, min_periods=1).mean()
        g['gust_max_6h'] = g['gust'].rolling(36, min_periods=1).max()
        g['gust_max_12h'] = g['gust'].rolling(72, min_periods=1).max()
        g['gust_minus_wspd'] = g['gust'] - g['wspd']

        g['caph_change_3h'] = g['caph'] - g['caph'].shift(18)
        g['caph_change_6h'] = g['caph'] - g['caph'].shift(36)
        g['caph_change_12h'] = g['caph'] - g['caph'].shift(72)

        wl = 1.56 * (g['tp'] ** 2)
        g['wave_steepness'] = g['hs'] / np.maximum(wl, 1.0)
        g['wave_energy'] = g['hs'] ** 2
        g['effective_wind_forcing'] = (g['wspd'] ** 2) * g['wind_wave_alignment']

        g = g.replace([np.inf, -np.inf], np.nan)
        g[BEST_FEATURES] = g[BEST_FEATURES].ffill().fillna(0.0)
        feature_frames.append(g)

    return pd.concat(feature_frames, ignore_index=True)

test_context = pd.read_parquet(TEST_CONTEXT_PATH)
test_index = pd.read_csv(TEST_INDEX_PATH)
test_features = add_test_features_exp26(test_context)

case_order = test_index['case_id'].drop_duplicates().tolist()
windows = [test_features[test_features['case_id'] == cid].sort_values('step_minute')[BEST_FEATURES].to_numpy(dtype=np.float32) for cid in case_order]
X_test_raw = np.stack(windows)
X_test = best_scaler.transform(X_test_raw.reshape(-1, len(BEST_FEATURES))).reshape(X_test_raw.shape).astype(np.float32)

final_model.eval()
preds = []
with torch.no_grad():
    for start in range(0, len(X_test), 128):
        xb = torch.from_numpy(X_test[start:start + 128]).to(DEVICE)
        preds.append(final_model(xb).detach().cpu().numpy())
preds = np.concatenate(preds, axis=0)

pred_rows = []
for case_id, row in zip(case_order, preds):
    for lead_h, val in zip(LEAD_HOURS, row):
        pred_rows.append({'case_id': case_id, 'lead_h': lead_h, 'hs_pred': float(val)})

pred_df = pd.DataFrame(pred_rows)
submission = test_index.merge(pred_df, on=['case_id', 'lead_h'], how='left', validate='one_to_one')
submission['hs_pred'] = submission['hs_pred'].clip(lower=0.05, upper=30.0)

submission.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8')
print('=' * 80)
print(f'★ EXP26 SUBMISSION SAVED: {SUBMISSION_PATH}')
print('=' * 80)
display(submission.head(12))
